# 🔬 Dual Model Audit & Clinical Validation Report

## Purpose
Comprehensive validation of **two distinct XGBoost models** for the Diabetes Decision Support System:

| Model | Task | Target |
|-------|------|--------|
| **Glucose Predictor** | Predict blood glucose 30 min ahead | Glucose (mg/dL) |
| **Insulin Recommender** | Recommend insulin bolus dose | Bolus (units) |

## Clinical Goals
- **Glucose Model**: RMSE < 20 mg/dL, Zone A+B > 95%
- **Insulin Model**: MAE < 2 units for positive doses

---

## 📦 Cell 0: Setup & Installation

In [ ]:
# ============================================================
# COLAB SETUP
# ============================================================
!pip install polars plotly xgboost -q
print("✅ Dependencies installed")

In [ ]:
# ============================================================
# MOUNT GOOGLE DRIVE
# ============================================================
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted")

In [ ]:
# ============================================================
# GPU CHECK
# ============================================================
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader
print("\n✅ GPU information displayed above")

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import os
import time
import polars as pl
import numpy as np
import xgboost as xgb
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print(f"✅ Polars: {pl.__version__}")
print(f"✅ XGBoost: {xgb.__version__}")

## ⚙️ Cell 1: Dynamic Path Configuration

In [ ]:
# ============================================================
# PATH CONFIGURATION - DUAL MODEL SETUP
# ============================================================

# Base Paths
BASE_DIR = "/content/drive/MyDrive/DiabetesKDS"
DATA_DIR = os.path.join(BASE_DIR, "data", "processed")

# File Maps for Both Models
FILES = {
    "glucose": {
        "name": "Glucose Prediction Model",
        "model": os.path.join(DATA_DIR, "xgboost_glucose_model_v2.json"),
        "data": os.path.join(DATA_DIR, "test_ready_for_glucose_prediction.parquet"),
        "target": "CGM_Target",
        "features": [
            'CGM', 'bolus', 'velocity', 'acceleration',
            'rolling_mean', 'rolling_std', 'active_insulin',
            'time_sin', 'time_cos', 'age', 'weight', 'gender_encoded'
        ]
    },
    "insulin": {
        "name": "Insulin Recommender Model",
        "model": os.path.join(DATA_DIR, "xgboost_final_model.json"),
        "data": os.path.join(DATA_DIR, "test_ready_for_xgboost.parquet"),
        "target": "bolus",
        "features": [
            'CGM', 'carbs', 'time_sin', 'time_cos', 'is_weekend',
            'cgm_delta_5min', 'cgm_delta_15min', 'basal',
            'age', 'weight', 'gender_encoded', 'insulin_type_encoded'
        ]
    }
}

# Clinical Thresholds
THRESHOLDS = {
    "glucose": {
        "hypo": 70,
        "hyper": 180,
        "rmse_goal": 20,
        "zone_ab_goal": 95
    },
    "insulin": {
        "dose_threshold": 0.5,
        "mae_goal": 2.0
    }
}

# GPU Inference Settings
GPU_CONFIG = {
    "batch_size": 500_000,  # Samples per batch for large datasets
    "device": "cuda",
    "tree_method": "hist"
}

# Validation
print("=" * 70)
print("📂 DUAL MODEL CONFIGURATION")
print("=" * 70)

for model_key, config in FILES.items():
    print(f"\n🔹 {config['name'].upper()}")
    print(f"   Model: {os.path.basename(config['model'])}")
    print(f"   Data:  {os.path.basename(config['data'])}")
    
    model_exists = os.path.exists(config['model'])
    data_exists = os.path.exists(config['data'])
    
    print(f"   {'✅' if model_exists else '❌'} Model file {'found' if model_exists else 'NOT FOUND'}")
    print(f"   {'✅' if data_exists else '❌'} Data file {'found' if data_exists else 'NOT FOUND'}")

# List available files for debugging
if os.path.exists(DATA_DIR):
    print(f"\n📁 Available files in {DATA_DIR}:")
    for f in sorted(os.listdir(DATA_DIR)):
        size = os.path.getsize(os.path.join(DATA_DIR, f)) / (1024*1024)
        print(f"   - {f} ({size:.1f} MB)")

## 🔧 Cell 2: Model Loading & GPU-Accelerated Inference

In [ ]:
# ============================================================
# MODEL LOADING & GPU-ACCELERATED INFERENCE ENGINE
# ============================================================

class ModelValidator:
    """Unified validator for XGBoost models with GPU acceleration."""
    
    def __init__(self, model_key: str):
        self.model_key = model_key
        self.config = FILES[model_key]
        self.model = None
        self.df = None
        self.y_true = None
        self.y_pred = None
        self.metrics = {}
        self.inference_time = 0
        
    def load_model(self):
        """Load XGBoost model from JSON."""
        print(f"\n📦 Loading {self.config['name']}...")
        self.model = xgb.Booster()
        self.model.load_model(self.config['model'])
        print(f"   ✅ Model loaded")
        return self
    
    def load_data(self):
        """Load test data from Parquet."""
        print(f"   📊 Loading test data...")
        self.df = pl.read_parquet(self.config['data'])
        print(f"   ✅ Data: {self.df.shape[0]:,} rows, {self.df.shape[1]} cols")
        print(f"   📋 Columns: {self.df.columns}")
        return self
    
    def run_inference(self):
        """Run model inference using GPU acceleration (CUDA)."""
        print(f"   🚀 Running inference (GPU Mode)...")
        
        # =============================================
        # FEATURE PREPARATION
        # =============================================
        available_features = [f for f in self.config['features'] if f in self.df.columns]
        missing_features = [f for f in self.config['features'] if f not in self.df.columns]
        
        if missing_features:
            print(f"   ⚠️ Missing features (filled with 0): {missing_features}")
            for f in missing_features:
                self.df = self.df.with_columns(pl.lit(0.0).alias(f))
            available_features = self.config['features']
        
        # =============================================
        # FORCE GPU/CUDA PARAMETERS (XGBoost 2.0+)
        # =============================================
        try:
            self.model.set_param({
                "device": GPU_CONFIG["device"],
                "tree_method": GPU_CONFIG["tree_method"]
            })
            print(f"   ✅ GPU mode enabled (device={GPU_CONFIG['device']})")
        except Exception as e:
            print(f"   ⚠️ GPU params failed: {e}")
            print(f"   ℹ️ Falling back to CPU mode")
        
        # =============================================
        # DATA PREPARATION
        # =============================================
        print(f"   📊 Preparing {self.df.shape[0]:,} samples...")
        t0 = time.time()
        X = self.df.select(available_features).to_numpy().astype(np.float32)
        X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
        print(f"   ✅ Data prepared in {time.time()-t0:.2f}s")
        
        # =============================================
        # BATCHED INFERENCE (FOR LARGE DATASETS 22M+)
        # =============================================
        n_samples = len(X)
        BATCH_SIZE = GPU_CONFIG["batch_size"]
        
        t0 = time.time()
        
        if n_samples > BATCH_SIZE:
            print(f"   ⚡ Batched inference: {n_samples:,} samples, batch={BATCH_SIZE:,}")
            predictions = []
            n_batches = (n_samples + BATCH_SIZE - 1) // BATCH_SIZE
            
            for batch_idx in range(n_batches):
                start_idx = batch_idx * BATCH_SIZE
                end_idx = min(start_idx + BATCH_SIZE, n_samples)
                batch = X[start_idx:end_idx]
                
                dmatrix = xgb.DMatrix(batch, feature_names=available_features)
                batch_pred = self.model.predict(dmatrix)
                predictions.append(batch_pred)
                
                # Progress
                progress_pct = (batch_idx + 1) / n_batches * 100
                print(f"      Batch {batch_idx+1}/{n_batches} ({progress_pct:.0f}%)      ", end='\r')
            
            self.y_pred = np.concatenate(predictions)
            print()  # New line after progress
        else:
            # Single batch for smaller datasets
            dmatrix = xgb.DMatrix(X, feature_names=available_features)
            self.y_pred = self.model.predict(dmatrix)
        
        self.inference_time = time.time() - t0
        print(f"   ✅ Inference complete in {self.inference_time:.2f}s")
        print(f"   📈 Throughput: {n_samples/self.inference_time:,.0f} samples/sec")
        
        # Clip negatives
        self.y_pred = np.maximum(self.y_pred, 0)
        
        # =============================================
        # GROUND TRUTH ALIGNMENT
        # =============================================
        target_col = self.config['target']
        if target_col in self.df.columns:
            self.y_true = self.df[target_col].to_numpy().astype(np.float32)
            valid_mask = ~np.isnan(self.y_true)
            self.y_true = self.y_true[valid_mask]
            self.y_pred = self.y_pred[valid_mask]
            print(f"   📈 Valid predictions: {len(self.y_pred):,}")
        else:
            print(f"   ⚠️ Target '{target_col}' not found - predictions only")
            self.y_true = None
            
        return self
    
    def calculate_metrics(self):
        """Calculate performance metrics."""
        if self.y_true is None:
            print("   ⚠️ No ground truth - skipping metrics")
            return self
        
        self.metrics['rmse'] = np.sqrt(mean_squared_error(self.y_true, self.y_pred))
        self.metrics['mae'] = mean_absolute_error(self.y_true, self.y_pred)
        self.metrics['r2'] = r2_score(self.y_true, self.y_pred)
        self.metrics['inference_time'] = self.inference_time
        
        # Positive-only metrics (for insulin)
        pos_mask = self.y_true > 0
        if pos_mask.sum() > 0:
            self.metrics['mae_positive'] = mean_absolute_error(
                self.y_true[pos_mask], self.y_pred[pos_mask]
            )
            self.metrics['n_positive'] = pos_mask.sum()
        
        print(f"\n   📈 METRICS:")
        print(f"      RMSE: {self.metrics['rmse']:.4f}")
        print(f"      MAE:  {self.metrics['mae']:.4f}")
        print(f"      R²:   {self.metrics['r2']:.4f}")
        if 'mae_positive' in self.metrics:
            print(f"      MAE (positive): {self.metrics['mae_positive']:.4f}")
        
        return self
    
    def run_full_pipeline(self):
        """Execute complete validation pipeline."""
        return self.load_model().load_data().run_inference().calculate_metrics()

print("✅ ModelValidator class defined (GPU-accelerated)")

---
# 🩸 Section 1: Glucose Prediction Model Validation

**Task**: Predict blood glucose 30 minutes ahead
**Clinical Importance**: Enables proactive interventions for hypo/hyperglycemia prevention

In [ ]:
# ============================================================
# GLUCOSE MODEL VALIDATION
# ============================================================

print("=" * 70)
print("🩸 GLUCOSE PREDICTION MODEL VALIDATION")
print("=" * 70)

glucose_validator = ModelValidator("glucose").run_full_pipeline()

In [ ]:
# ============================================================
# GLUCOSE: CLARKE ERROR GRID
# ============================================================

def clarke_error_zone(y_actual, y_pred):
    """Determine Clarke Error Grid zone (vectorized)."""
    zones = np.empty(len(y_actual), dtype='U1')
    
    for i in range(len(y_actual)):
        ref, pred = y_actual[i], y_pred[i]
        
        if (ref <= 70 and pred <= 70) or (ref >= 70 and abs(pred - ref) <= 0.2 * ref):
            zones[i] = 'A'
        elif (ref >= 180 and pred <= 70) or (ref <= 70 and pred >= 180):
            zones[i] = 'E'
        elif (ref < 70 and pred >= 70 and pred < 180) or (ref > 180 and pred > 70 and pred <= 180):
            zones[i] = 'D'
        elif (ref >= 70 and ref <= 180 and pred < 70) or (ref >= 70 and ref <= 180 and pred > 180):
            zones[i] = 'C'
        else:
            zones[i] = 'B'
    
    return zones

if glucose_validator.y_true is not None:
    zones = clarke_error_zone(glucose_validator.y_true, glucose_validator.y_pred)
    zone_counts = {z: np.sum(zones == z) for z in ['A', 'B', 'C', 'D', 'E']}
    total = len(zones)
    
    print("\n📊 CLARKE ERROR GRID ANALYSIS")
    print("=" * 50)
    for zone, count in zone_counts.items():
        pct = count / total * 100
        bar = '█' * int(pct / 2)
        print(f"Zone {zone}: {count:6,} ({pct:5.2f}%) {bar}")
    
    zone_ab_pct = (zone_counts['A'] + zone_counts['B']) / total * 100
    goal = THRESHOLDS['glucose']['zone_ab_goal']
    print(f"\n{'✅' if zone_ab_pct >= goal else '⚠️'} Zone A+B: {zone_ab_pct:.2f}% (Target: >{goal}%)")
    
    glucose_validator.metrics['zone_ab_pct'] = zone_ab_pct

In [ ]:
# ============================================================
# GLUCOSE: VISUALIZATION
# ============================================================

if glucose_validator.y_true is not None:
    fig = make_subplots(rows=1, cols=2, subplot_titles=(
        'Predicted vs Actual Glucose', 'Error Distribution'
    ))
    
    # Sample for visualization (avoid plotting 22M points)
    sample_size = min(50000, len(glucose_validator.y_true))
    idx = np.random.choice(len(glucose_validator.y_true), sample_size, replace=False)
    
    fig.add_trace(go.Scatter(
        x=glucose_validator.y_true[idx],
        y=glucose_validator.y_pred[idx],
        mode='markers',
        marker=dict(size=3, opacity=0.3, color='#3498db'),
        name='Predictions'
    ), row=1, col=1)
    
    max_val = max(glucose_validator.y_true.max(), glucose_validator.y_pred.max())
    fig.add_trace(go.Scatter(
        x=[0, max_val], y=[0, max_val],
        mode='lines', line=dict(color='red', dash='dash'),
        name='Perfect'
    ), row=1, col=1)
    
    errors = glucose_validator.y_pred - glucose_validator.y_true
    fig.add_trace(go.Histogram(
        x=errors, nbinsx=50,
        marker_color='#2ecc71', opacity=0.7,
        name='Error'
    ), row=1, col=2)
    
    fig.update_layout(
        title=f'🩸 Glucose Model (n={len(glucose_validator.y_true):,}, sampled {sample_size:,})',
        height=400, showlegend=False, template='plotly_white'
    )
    fig.update_xaxes(title_text='Actual (mg/dL)', row=1, col=1)
    fig.update_yaxes(title_text='Predicted (mg/dL)', row=1, col=1)
    fig.update_xaxes(title_text='Prediction Error (mg/dL)', row=1, col=2)
    
    fig.show()

---
# 💉 Section 2: Insulin Recommender Model Validation

**Task**: Recommend optimal bolus insulin dose
**Clinical Importance**: Accurate dosing prevents both hypoglycemia and hyperglycemia

In [ ]:
# ============================================================
# INSULIN MODEL VALIDATION
# ============================================================

print("\n" + "=" * 70)
print("💉 INSULIN RECOMMENDER MODEL VALIDATION")
print("=" * 70)

insulin_validator = ModelValidator("insulin").run_full_pipeline()

In [ ]:
# ============================================================
# INSULIN: CLINICAL SAFETY ANALYSIS
# ============================================================

if insulin_validator.y_true is not None:
    threshold = THRESHOLDS['insulin']['dose_threshold']
    
    pos_mask = insulin_validator.y_true > threshold
    n_positive = pos_mask.sum()
    
    missed_mask = pos_mask & (insulin_validator.y_pred < threshold)
    n_missed = missed_mask.sum()
    miss_rate = n_missed / n_positive * 100 if n_positive > 0 else 0
    
    print("\n🏥 CLINICAL SAFETY ANALYSIS")
    print("=" * 50)
    print(f"   Total samples:        {len(insulin_validator.y_true):,}")
    print(f"   Positive doses:       {n_positive:,} ({n_positive/len(insulin_validator.y_true)*100:.1f}%)")
    print(f"   Missed detections:    {n_missed:,}")
    print(f"   Miss Rate:            {miss_rate:.2f}%")
    
    goal = THRESHOLDS['insulin']['mae_goal']
    pos_mae = insulin_validator.metrics.get('mae_positive', np.nan)
    print(f"\n{'✅' if pos_mae <= goal else '⚠️'} Positive-Only MAE: {pos_mae:.4f} (Target: <{goal})")
    
    insulin_validator.metrics['miss_rate'] = miss_rate

In [ ]:
# ============================================================
# INSULIN: VISUALIZATION
# ============================================================

if insulin_validator.y_true is not None:
    fig = make_subplots(rows=1, cols=2, subplot_titles=(
        'Predicted vs Actual Dose (Positive Only)', 'Feature Importance'
    ))
    
    pos_mask = insulin_validator.y_true > 0
    
    # Sample positive cases for visualization
    pos_indices = np.where(pos_mask)[0]
    sample_size = min(50000, len(pos_indices))
    sampled_idx = np.random.choice(pos_indices, sample_size, replace=False)
    
    fig.add_trace(go.Scatter(
        x=insulin_validator.y_true[sampled_idx],
        y=insulin_validator.y_pred[sampled_idx],
        mode='markers',
        marker=dict(size=4, opacity=0.4, color='#e74c3c'),
        name='Predictions'
    ), row=1, col=1)
    
    max_dose = max(insulin_validator.y_true[pos_mask].max(), 
                   insulin_validator.y_pred[pos_mask].max())
    fig.add_trace(go.Scatter(
        x=[0, max_dose], y=[0, max_dose],
        mode='lines', line=dict(color='green', dash='dash'),
        name='Perfect'
    ), row=1, col=1)
    
    importance = insulin_validator.model.get_score(importance_type='gain')
    if importance:
        sorted_imp = sorted(importance.items(), key=lambda x: x[1], reverse=True)
        features, gains = zip(*sorted_imp[:10])
        
        fig.add_trace(go.Bar(
            x=list(gains),
            y=list(features),
            orientation='h',
            marker_color='#9b59b6'
        ), row=1, col=2)
    
    fig.update_layout(
        title=f'💉 Insulin Model (pos={pos_mask.sum():,}, sampled {sample_size:,})',
        height=400, showlegend=False, template='plotly_white'
    )
    fig.update_xaxes(title_text='Actual Dose (units)', row=1, col=1)
    fig.update_yaxes(title_text='Predicted Dose (units)', row=1, col=1)
    fig.update_xaxes(title_text='Gain', row=1, col=2)
    
    fig.show()

---
# 📋 Section 3: Combined Summary Report

In [ ]:
# ============================================================
# FINAL COMBINED REPORT
# ============================================================

print("\n" + "=" * 70)
print("📋 DUAL MODEL AUDIT SUMMARY")
print("=" * 70)

# Glucose Model Summary
print("\n🩸 GLUCOSE PREDICTION MODEL")
print("-" * 40)
g = glucose_validator.metrics
print(f"   RMSE:      {g.get('rmse', 'N/A'):.2f} mg/dL")
print(f"   MAE:       {g.get('mae', 'N/A'):.2f} mg/dL")
print(f"   R²:        {g.get('r2', 'N/A'):.4f}")
print(f"   Inference: {g.get('inference_time', 0):.2f}s")
if 'zone_ab_pct' in g:
    goal_g = THRESHOLDS['glucose']['zone_ab_goal']
    status_g = '✅ PASS' if g['zone_ab_pct'] >= goal_g else '❌ FAIL'
    print(f"   Zone A+B:  {g['zone_ab_pct']:.2f}% {status_g}")

# Insulin Model Summary
print("\n💉 INSULIN RECOMMENDER MODEL")
print("-" * 40)
i = insulin_validator.metrics
print(f"   RMSE:      {i.get('rmse', 'N/A'):.4f} units")
print(f"   MAE:       {i.get('mae', 'N/A'):.4f} units")
print(f"   R²:        {i.get('r2', 'N/A'):.4f}")
print(f"   Inference: {i.get('inference_time', 0):.2f}s")
if 'mae_positive' in i:
    goal_i = THRESHOLDS['insulin']['mae_goal']
    status_i = '✅ PASS' if i['mae_positive'] <= goal_i else '❌ FAIL'
    print(f"   MAE (pos): {i['mae_positive']:.4f} units {status_i}")
if 'miss_rate' in i:
    print(f"   Miss Rate: {i['miss_rate']:.2f}%")

print("\n" + "=" * 70)
print(f"Report generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 70)

In [ ]:
# ============================================================
# SAVE RESULTS (OPTIONAL)
# ============================================================

SAVE_RESULTS = False  # Set to True to save

if SAVE_RESULTS:
    output_dir = os.path.join(BASE_DIR, "outputs", "audit_reports")
    os.makedirs(output_dir, exist_ok=True)
    
    combined = {
        'model': ['glucose', 'insulin'],
        'rmse': [g.get('rmse'), i.get('rmse')],
        'mae': [g.get('mae'), i.get('mae')],
        'r2': [g.get('r2'), i.get('r2')],
        'inference_time': [g.get('inference_time'), i.get('inference_time')]
    }
    pl.DataFrame(combined).write_csv(os.path.join(output_dir, "dual_model_metrics.csv"))
    print(f"✅ Results saved to: {output_dir}")